# Notebook 01: Literature Review — TALE-Fusion Geometric Constraints

**Paper C5:** *Geometric Constraints on Catalytic Domain Fusion to TALE Arrays*

This notebook summarises the literature survey of published TALE-fusion and CRISPR-fusion
architectures, focusing on linker lengths, fusion geometries, and performance metrics.

Key references:
- Miller et al. 2011 (TALEN, C+63 architecture)
- Beurdeley et al. 2013 (compact TALEN / I-TevI)
- Yamano et al. 2013 (TALE-PvuII geometry)
- Mok et al. 2020 (DdCBE mitochondrial base editor)

**Central insight:** The optimal linker length is determined by the *target distance*,
not by maximising flexibility. This is the *length-matching principle*.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# Load literature survey
df = pd.read_csv('../data/literature_survey.csv')
print(f'Literature survey: {len(df)} papers')
df[['Paper / Source', 'Year', 'Fusion Type', 'Catalytic Domain', 'Linker Length']].head(10)

In [ ]:
# Parse linker lengths (some are ranges like '40-63')
def parse_length(val):
    if pd.isna(val) or str(val).strip() in ('', 'Variable', 'N/A'):
        return None
    val = str(val)
    if '-' in val:
        parts = val.split('-')
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except ValueError:
            return None
    try:
        return float(''.join(c for c in val if c.isdigit() or c == '.'))
    except ValueError:
        return None

df['Length_num'] = df['Linker Length'].apply(parse_length)
df_with_length = df[df['Length_num'].notna()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: linker length distribution
axes[0].hist(df_with_length['Length_num'], bins=8, color='#4A90D9', edgecolor='white', linewidth=1.2)
axes[0].axvline(10, color='#E74C3C', lw=2, ls='--', label='GENESIS Design 1 (10 res)')
axes[0].set_xlabel('Linker Length (residues)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Published TALE-Fusion Linker Lengths', fontsize=13, fontweight='bold')
axes[0].legend()

# Right: fusion type breakdown
type_counts = df['Fusion Type'].value_counts()
axes[1].barh(range(len(type_counts)), type_counts.values, 
             color=['#4A90D9', '#5DADE2', '#76C1D3','#A9CCE3','#D6E8F3'][:len(type_counts)])
axes[1].set_yticks(range(len(type_counts)))
axes[1].set_yticklabels(type_counts.index)
axes[1].set_xlabel('Number of Papers', fontsize=12)
axes[1].set_title('Fusion Architecture Types\nin Literature Survey', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/supp_literature_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved supplementary literature overview figure.')

In [ ]:
# Key table: linker geometries of benchmark published fusions
key_cols = ['Paper / Source', 'Year', 'Fusion Type', 'Catalytic Domain', 'Linker Architecture Details', 
            'Linker Length', 'Reported Efficiency']
print('Key published TALE and CRISPR fusions:')
pd.set_option('display.max_colwidth', 60)
df[key_cols].dropna(subset=['Fusion Type'])

## Key Findings from Literature Survey

1. **Linker length range:** Published TALE-fusion linkers span 4–63 residues.
   Most use long flexible sequences (>40 res) that dramatically oversample space.

2. **The compact TALEN paradox (Beurdeley 2013):** The I-TevI linker (~59 res)
   achieves high specificity *not* through shortness, but through a second recognition
   layer (CNNNG sequence gating). Linker flexibility alone does not determine specificity.

3. **Geometric gap:** No published work provides a first-principles geometric analysis
   of the linker length required to bridge the TALE C-terminus to a specific scissile
   phosphate. This gap is what the C5 paper addresses.

4. **The length-matching principle:** Our WLC analysis shows that P(reach) is
   maximised when the mean end-to-end distance of the linker ≈ the target distance.
   For the GENESIS primary target (bp +4, 16 Å), this predicts n ≈ 10 residues
   for helix, 10 for GGS — converging independently on the same optimal length.